# 실험 안내

각 노트북은 질문과 정의, 관련 코드, 결과 해석 순서로 읽습니다. 설명과 수식은 해당 코드 근처의 Markdown 셀에 있습니다.

## 실험 목록

- [01_objective](01_objective.ipynb): 논문이 쓴 목적함수와 알고리즘이 만드는 해
- [02_chunk_latency](02_chunk_latency.ipynb): throughput 99% 첫 도달 길이 s와 chunk latency 모델 검증
- [03_global_oracle](03_global_oracle.ipynb): 작은 채널 공간에서의 global solution
- [04_polynomial_dp](04_polynomial_dp.ipynb): 가산 run 모델의 다항시간 해법과 한계
- [05_measured_frontier](05_measured_frontier.ipynb): 새로 측정한 latency 위의 importance–latency 비교
- [06_saturation_guarantee](06_saturation_guarantee.ipynb): 포화 길이 활용 알고리즘: 보장되는 부분과 깨지는 부분
- [07_mask_walltime](07_mask_walltime.ipynb): mask의 예측 latency와 실제 읽기 시간
- [08_importance_error](08_importance_error.ipynb): importance는 무엇을 보장하는가: 새 activation과 실제 projection 오차
- [09_global_allocation](09_global_allocation.ipynb): 고정 sparsity 대신 전역 importance 하한으로 예산 배분

## 실행

저장소 루트 또는 `conclusion/python/`에서 노트북을 열고 Run All을 실행합니다. 전체 또는 선택한 실험을 새 커널에서 실행할 수도 있습니다.

```bash
python -m pip install -r conclusion/python/requirements.txt
OPENBLAS_NUM_THREADS=1 python conclusion/python/run_all.py
OPENBLAS_NUM_THREADS=1 python conclusion/python/run_all.py 02 07
python conclusion/python/check_algorithms.py
```

02의 전체 SSD sweep은 시간이 걸립니다. 02는 7개 반복 블록 중 4개로 적합하고 3개로 검증합니다. 개정 후 실험은 실행하지 않았으며, 02의 이전 프로토콜 출력은 노트북에서 제거했습니다. 디스크의 이전 결과도 새 검증 결과로 해석하지 않습니다.

## 결과 저장

각 실험은 `runs/01_objective/`, `runs/02_chunk_latency/`처럼 고정 폴더를 사용합니다. 실행 준비 셀의 `begin()`이 그 실험의 이전 폴더를 지우고 새로 만듭니다. 다른 실험의 폴더는 건드리지 않습니다.

- `settings.json`: 난수 seed.
- `hardware.json`, `model.json`: 해당 측정의 장치 설정 또는 사용한 모델 정보.
- CSV/NPZ: 측정값과 계산 결과.
- PNG/SVG: 그림.
- `findings.json`: 마지막 결과 요약.

결과 파일은 다음 실험의 입력으로 사용하지 않습니다. 이전 결과를 남기려면 다시 실행하기 전에 필요한 파일을 별도로 복사합니다.

## 필요한 환경

- 02는 `conclusion/upstream/vlm-flash` 서브모듈의 native C++ profiler를 사용합니다. C++ 컴파일러와 Ninja가 필요하며 첫 실행에서 PyTorch extension을 빌드합니다. 128 MiB 임시 파일에서 1–768 KiB의 384개 크기를 7개 블록에서 측정하고 `tqdm` 진행바 하나로 진행 상황을 표시합니다.
- 05·07·09는 Linux `O_DIRECT`와 `preadv`로 프로젝트가 위치한 파일시스템을 측정합니다. 최대 512 MiB 임시 파일을 만들고 측정 후 삭제합니다.
- 08·09는 로컬 `SmolLM2-360M-Instruct` checkpoint가 필요합니다. 기본 revision은 `a10cc1512eabd3dde888204e902eca88bddb4951`이며 `VLM_CHECKPOINT=/path/to/snapshot`으로 경로를 지정할 수 있습니다. 자동 다운로드나 synthetic 대체를 하지 않습니다.
- 측정 중 다른 I/O 작업을 피하고, Jupyter 실행에는 로컬 커널 소켓을 허용합니다.

## 공통 코드

- [core.py](core.py): selector, oracle, 두 DP.
- [measure.py](measure.py): Python direct I/O 측정.
- [capture.py](capture.py): 로컬 모델 activation 수집.
- [reporting.py](reporting.py): 결과 폴더 초기화, 그림과 요약 저장.
- [run_all.py](run_all.py): 선택한 노트북을 새 커널에서 실행.

수학 모델의 정확해, synthetic 입력 비교, SSD 실측, projection 오차는 각 노트북의 가정과 범위에 따라 해석합니다. 전체 모델 추론 시간이나 VLM task accuracy를 측정한 것으로 확대하지 않습니다.